# Aerofit Treadmill Customer & Product Exploratory Data Analysis (EDA)

**Author:** Portfolio Data Science Project  
**Dataset:** `aerofit_treadmill.csv` (180 customer purchase and usage records across KP281, KP481, and KP781 treadmills)  

### Objective
Analyze customer demographics, fitness habits, weekly usage, and income levels to uncover purchase drivers across Aerofit's three treadmill models and deliver actionable marketing and product positioning recommendations.

## 1. Environment Setup & Data Ingestion

In [ ]:
# Import core data manipulation, numerical, and visualization libraries
import numpy as np              # Numerical operations and conditional vector selection (np.select)
import pandas as pd             # DataFrames, aggregations, crosstabs, and binning utilities
import matplotlib.pyplot as plt  # Base plotting library for figure sizing, labels, and layout control
import seaborn as sns           # High-level statistical visualization built on Matplotlib
import gdown                    # Utility to download public files directly in Google Colab

# Configure global visualization aesthetics for clean, consistent portfolio charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Define remote dataset URL and local filename for Google Colab execution
url = "https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/001/125/original/aerofit_treadmill.csv?1639992749"
output_file = "aerofit_treadmill.csv"

# Download the CSV file from CloudFront to the Colab runtime environment
gdown.download(url, output_file, quiet=False)

# Load the dataset into a Pandas DataFrame
df = pd.read_csv(output_file)

# Preview the first 5 rows to verify column names and sample values
df.head()

## 2. Structural Inspection & Feature Engineering

We inspect data types, dimensions, missing values, and exact row counts first, then engineer **Income Brackets** and **Age Brackets** right away so downstream EDA cells have access to these segmented features.

In [ ]:
# Check dataset dimensions (rows, columns)
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

# Display column data types and non-null counts to confirm zero missing values
print("--- DataFrame Info ---")
df.info()

# Explicitly verify missing values and duplicate rows as a data quality check
print("\nMissing Values per Column:")
print(df.isnull().sum())
print(f"Duplicate Rows: {df.duplicated().sum()}")

In [ ]:
# 1. Engineer 'income_range' using vectorized np.select conditions on annual Income (USD)
# Note: Assigning directly to df['income_range'] preserves exact row alignment with df's index
income_conditions = [
    df["Income"] >= 60000,
    (df["Income"] >= 30000) & (df["Income"] < 60000),
    df["Income"] < 30000
]
income_labels = ["high", "medium", "low"]
df["income_range"] = np.select(income_conditions, income_labels, default="unknown")

# 2. Engineer 'Age_Bracket' by binning continuous Age values into interpretable life/fitness stages
age_bins = [18, 25, 35, 50]
age_labels = ["18-25", "26-35", "36-50"]
df["Age_Bracket"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True  # Ensures minimum age (18) is included in the first bin
)

# Confirm new engineered features are present alongside original columns
df[["Age", "Age_Bracket", "Income", "income_range", "Product"]].head()

## 3. Statistical Summary

In [ ]:
# Generate summary statistics (count, mean, std, min, quartiles, max) for numerical columns
df.describe().T

### Statistical Summary Takeaways
- **Sample Size:** 180 customer records with no missing entries.
- **Age:** Mean ~28.8 years (range: 18–50), skewed slightly right toward younger-to-early-career adults.
- **Education:** Mean ~15.6 years (range: 12–21), indicating college-educated buyers on average.
- **Weekly Usage (`Usage`):** Mean ~3.46 sessions/week (range: 2–7).
- **Self-Rated Fitness (`Fitness`):** Mean ~3.31 on a 1–5 scale (range: 1–5).
- **Annual Income (`Income`):** Mean ~$53,720 USD with std ~$16,507 USD (range: $29,562–$104,581).
- **Weekly Distance (`Miles`):** Mean ~103.19 miles/week (range: 21–360).

## 4. Univariate Analysis (Demographics, Behavior & Outliers)

In [ ]:
# Compute raw frequency counts for Gender
gender_count = df["Gender"].value_counts()
print("Gender Counts:")
print(gender_count)

# Plot pie chart comparing Male vs. Female share with slight slice separation
plt.figure(figsize=(7, 5))
plt.pie(
    gender_count,
    labels=gender_count.index,
    autopct="%1.1f%%",
    startangle=90,
    explode=[0.05] * len(gender_count),
    colors=sns.color_palette("pastel", n_colors=len(gender_count))
)
plt.title("Customer Distribution by Gender")
plt.tight_layout()
plt.show()

In [ ]:
# Compute central tendency metrics and mean-median skew check for Age
age_mean = df["Age"].mean()
age_median = df["Age"].median()
print(f"Age -> Mean: {age_mean:.2f}, Median: {age_median:.2f}, Mean - Median: {age_mean - age_median:.2f}")

# Create side-by-side histogram (with KDE) and boxplot to inspect distribution shape and outliers
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(data=df, x="Age", bins=10, kde=True, ax=axes[0])
axes[0].axvline(age_mean, color="red", linestyle="--", linewidth=1.2, label=f"Mean ({age_mean:.1f})")
axes[0].axvline(age_median, color="green", linestyle=":", linewidth=1.5, label=f"Median ({age_median:.1f})")
axes[0].set_title("Age Distribution (Histogram + KDE)")
axes[0].set_xlabel("Age (Years)")
axes[0].set_ylabel("Customer Count")
axes[0].legend()

sns.boxplot(x=df["Age"], ax=axes[1])
axes[1].set_title("Age Boxplot (Outlier Detection)")
axes[1].set_xlabel("Age (Years)")

plt.tight_layout()
plt.show()

In [ ]:
# Count partnered/married vs. single customers directly from MaritalStatus
marital_counts = df["MaritalStatus"].value_counts()

# Plot share of Partnered vs. Single customers
plt.figure(figsize=(7, 5))
plt.pie(
    marital_counts.values,
    labels=marital_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    explode=[0.05 if status == "Single" else 0 for status in marital_counts.index],
    colors=["#FFA07A", "#87CEFA"]
)
plt.axis("equal")  # Keep pie circular rather than elliptical
plt.title("Customer Distribution by Marital Status")
plt.tight_layout()
plt.show()

## 5. Bivariate Relationships, Product Mix & Conditional Probabilities

In [ ]:
# Calculate marginal purchase probability (%) for each treadmill model
marginal_prob = pd.crosstab(index=df["Product"], columns="percentage", normalize=True) * 100
print("Marginal Probability of Product Purchases (%):")
print(marginal_prob.round(2))

# Visualize product share with a donut/pie chart
labels = marginal_prob.index.tolist()
sizes = marginal_prob["percentage"].tolist()

plt.figure(figsize=(6.5, 6.5))
plt.pie(
    sizes,
    labels=labels,
    colors=["#FDB813", "#F77737", "#CD525C"],
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.82
)
plt.title("Marginal Share of Treadmill Purchases by Model")
plt.tight_layout()
plt.show()

In [ ]:
# Plot key demographic and income relationships across treadmill models in a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Income distribution segmented by Product
sns.histplot(data=df, x="Income", hue="Product", kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Income Distribution by Product")
axes[0, 0].set_xlabel("Annual Income (USD)")

# 2. Marital Status count by Product
sns.countplot(data=df, x="MaritalStatus", hue="Product", ax=axes[0, 1])
axes[0, 1].set_title("Product Purchases by Marital Status")
axes[0, 1].set_xlabel("Marital Status")
axes[0, 1].set_ylabel("Customer Count")

# 3. Age Bracket count by Product
sns.countplot(data=df, x="Age_Bracket", hue="Product", ax=axes[1, 0])
axes[1, 0].set_title("Product Purchases by Age Bracket")
axes[1, 0].set_xlabel("Age Bracket")
axes[1, 0].set_ylabel("Customer Count")

# 4. Age distribution across Product models via boxplot
sns.boxplot(data=df, x="Product", y="Age", ax=axes[1, 1])
axes[1, 1].set_title("Age Distribution Across Treadmill Models")
axes[1, 1].set_xlabel("Product Model")
axes[1, 1].set_ylabel("Age (Years)")

plt.tight_layout()
plt.show()

In [ ]:
# Fixed filter: filter specifically for 'KP781' (previously filtered 'KP281' with a 'KP781' comment)
kp781_df = df[df["Product"] == "KP781"]

# Compute P(Gender == 'Male' | Product == 'KP781')
male_kp781_count = (kp781_df["Gender"] == "Male").sum()
total_kp781_count = len(kp781_df)
prob_male_given_kp781 = male_kp781_count / total_kp781_count

print(f"P(Male | KP781) = {male_kp781_count}/{total_kp781_count} = {prob_male_given_kp781:.4f}")

# Row-normalized contingency tables: P(Product | Gender) and P(Product | income_range)
gender_product_prob = pd.crosstab(df["Gender"], df["Product"], normalize="index")
income_product_prob = pd.crosstab(df["income_range"], df["Product"], normalize="index")

print("\nConditional Probability of Product Given Gender (Row-Normalized):")
print(gender_product_prob.round(4))

print("\nConditional Probability of Product Given Income Range (Row-Normalized):")
print(income_product_prob.round(4))

In [ ]:
# Select only numeric columns before computing Pearson correlation to avoid dtype errors
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

# Plot correlation heatmap for all numeric features
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Correlation Matrix of Numeric Customer & Usage Features")
plt.tight_layout()
plt.show()

In [ ]:
# Verify key portfolio & segment metrics across the 180-customer dataset
total_customers = len(df)

# 1. Gender distribution across all AeroFit treadmill buyers
gender_share = df["Gender"].value_counts(normalize=True) * 100
print(f"Total Customers: {total_customers}")
print("Gender Distribution (%):")
print(gender_share.round(2))

# 2. Age 21-30 preference for 'KP281'
age_21_30 = df[(df["Age"] >= 21) & (df["Age"] <= 30)]
kp281_share_21_30 = (age_21_30["Product"] == "KP281").mean() * 100
print(f"\nAge 21-30 Count: {len(age_21_30)} | KP281 Share in Age 21-30: {kp281_share_21_30:.2f}%")
print(pd.crosstab(age_21_30["Age"], age_21_30["Product"], normalize="index").round(3) * 100)

# 3. Income < $60,000 preference for 'KP281' and 'KP481'
under_60k = df[df["Income"] < 60000]
kp281_kp481_share_under_60k = under_60k["Product"].isin(["KP281", "KP481"]).mean() * 100
print(f"\nIncome < $60k Count: {len(under_60k)} | KP281 + KP481 Combined Share (<$60k): {kp281_kp481_share_under_60k:.2f}%")
print(under_60k["Product"].value_counts(normalize=True).round(4) * 100)

## 6. Key Findings & Actionable Recommendations for AeroFit

### Key Findings
1. **Demographic & Gender Mix:** Across the 180 AeroFit treadmill buyers, males account for the majority share (~57.8%) and females account for ~42.2%, with distinct model affinities across genders and lifestyle stages.
2. **Young-to-Early-Career Demand (`KP281`):** Younger-to-early-career buyers (such as the 21–30 age cohort) show strong demand for the entry-level `KP281` treadmill, driven by accessible pricing and straightforward everyday cardio usability.
3. **Income-Tier Product Segmentation (`KP281` & `KP481` vs. `KP781`):** Customers earning below $60,000 annually predominantly choose entry- and mid-tier models (`KP281` and `KP481`), whereas higher-income buyers gravitate more strongly toward the premium `KP781` model alongside higher weekly mileage and fitness levels.

### Strategic & Marketing Recommendations
- **Prioritize `KP281` & `KP481` for Under-$60k and Core Entry Segments:** Emphasize affordability, value for money, reliability, and accessible home-fitness routines for customers earning under $60,000 and first-time treadmill buyers.
- **Position `KP281` Effectively for Young Adults (Ages 21–30):** Highlight compact design, ease of use, and budget-friendly ownership for early-career and active young-adult buyers.
- **Target Higher-Income & Performance Segments with `KP781`:** Focus premium positioning—highlighting advanced features, durability, and high-mileage/high-fitness performance—on higher-income customers and serious runners, validating campaign changes through targeted A/B tests.